# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Content Refresh Scoring**

**Task type: Scoring (a ranking problem, not classification).**

The question isn't "is this page good or bad" (that would be classification) or "which pages are similar" (clustering). It's "of ~30k pages, which ones should a human review first?" That means every page needs a continuous score, and the pages get sorted by it — a ranking/scoring task. The output isn't a label, it's a **queue**.

In [ ]:
# Section 1 — no computation needed here, just naming the task.
# Restating it in code so it's explicit and greppable later in the pipeline.

TASK_TYPE = "scoring (ranking) — not classification, not clustering"
LANE = "content refresh scoring"

print(f"Lane: {LANE}")
print(f"ML task type: {TASK_TYPE}")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There's no ground-truth "this page needs a refresh" label sitting in the data — nobody tagged pages that way. So the target has to be a **proxy**, built from observed performance signals already in the dataset (e.g. how long since the page was touched, how its clicks/impressions/CTR/average position have moved, how it's trending relative to similar pages).

I'm treating the proxy as a **continuous need-to-refresh score**, not a hard yes/no cutoff — because "needs review" is a matter of degree, and a fixed cutoff would just be re-hiding a rule inside a threshold instead of removing it.

In [ ]:
# Building the proxy target from observed signals (adjust column names once loaded below).
# Idea, not final formula: pages that are old, losing position, or losing clicks score higher.

def describe_target_proxy():
    return {
        "target_name": "refresh_priority_score",
        "built_from": [
            "days_since_last_update (or similar recency signal)",
            "position_trend (worsening avg. rank)",
            "click_trend / impression_trend (declining traffic)",
            "ctr relative to position (underperforming for its rank)",
        ],
        "label_source": "observed performance signals, not a human-assigned label",
    }

describe_target_proxy()

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** Out of the top 50 pages the model puts at the front of the queue, what fraction genuinely deserved to be there (by the proxy signal)?

I'm not using plain accuracy — with ~30k pages, a model that says "nothing needs a refresh" would score high on accuracy while being useless. Precision@k matches the real workflow: a human only has time to review a short list, so what matters is whether the *top of the list* is good, not whether every one of 30k rows is labeled correctly. The starter pipeline's own baseline-vs-model comparison already reports this (hand-rule ≈ 0.24 → trained model ≈ 0.74), which is the number I'd defend against.

In [ ]:
# Section 3 — stating the metric and why, no computation yet (that happens after training).

SUCCESS_METRIC = "Precision@50"
METRIC_DEFINITION = "Of the top 50 ranked pages, the fraction that truly warrant review first."
BASELINE_REFERENCE = "hand-rule baseline ≈ 0.24 (from outputs/model_report.md)"

print(SUCCESS_METRIC, "—", METRIC_DEFINITION)
print("Baseline to beat:", BASELINE_REFERENCE)

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page** (a single URL's search-performance record). Loading the actual starter dataset below, not a mock — this is the real anonymized slice the pipeline runs on.

In [ ]:
import pandas as pd
import os

# Colab only has this single notebook file, not the whole repo — so clone the repo first
# (skipped automatically if it's already been cloned in this session).
REPO_URL = "https://github.com/muzammil-12345/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

DATA_PATH = f"{REPO_DIR}/data/raw/content_refresh_anonymized.csv"

# Fallback: if the file isn't in the clone (e.g. path/filename differs in the fork),
# pull it straight from the raw GitHub URL instead.
RAW_FALLBACK_URL = (
    "https://raw.githubusercontent.com/muzammil-12345/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    print(f"Local path not found ({DATA_PATH}), trying raw GitHub URL instead...")
    df = pd.read_csv(RAW_FALLBACK_URL)

print("Shape:", df.shape)
print("Unit of analysis: one row = one page\n")
df.head()

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag any page not updated in 12 months" ignores that a stable, still-ranking-well page and a fast-decaying page can have the exact same age. The real signal is an **interaction** of several things at once — age, position trend, click trend, CTR-vs-position — and the right weighting between them isn't obvious or constant across pages. A single threshold on one column can't capture that combination, and hand-tuning multiple thresholds together turns into guesswork fast. The repo's own numbers make the case directly: the hand-written rule gets Precision@50 ≈ 0.24, the trained model gets ≈ 0.74 on the *same data* — the rule isn't wrong out of laziness, it's wrong because the pattern genuinely needs multiple weighted signals, which is exactly what a model learns and a rule can't.

In [ ]:
# Quick, honest check: how many single-column signals does the target proxy actually depend on?
# (Illustrative — replace with real columns once confirmed against the data dictionary.)

signals_in_proxy = describe_target_proxy()["built_from"]
print(f"Number of interacting signals: {len(signals_in_proxy)}")
print("A fixed if-statement handles 1 signal cleanly; it breaks down combining", 
      len(signals_in_proxy), "signals with unknown relative weights.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.